# 05 pycavi Scamp

This notebook run SExtractor and Scamp over the mosaic images resulting from WCS solution retrieves by solve-field in notebook 4.

The flow is as follows:
- For each image with a wcs mosaic will first run SExtractor to create a FITS_LDAC catalogue of the stars in the field.
- Once the catalogue is created, Scamp is called. It will first extract the headers of the mosaic to create a .ahead file which will additionaly include airmass and filter header. Scamp will be called over the FITS_LDAC catalogue creating a .head file with the improved WCS headers.

Tested on my laptop, takes approximately 1 minute by mosaic. <br>
Tested with SExtractor 2.25 and Scamp 2.0.4. Please consider that older versions of SExtractor are called with /usr/bin/sex, but the newest (the one used here) are called with /usr/bin/source-extractor. I remember that in Chapman SExtractor must be called in the old way, so you will have to modified the SExtractor.run method to call the program correctly. The differences also includes the SExtractor parameters. Some parameters from the original file (for the older version) rises troubles when included in the new version. I have not tested if the 2.25 configuration works well on older versions.
If needed, I also include the old parameter configuration.

All the functions that relates with SExtractor or Scamp, like the creation of the configuration files, and the execution of the programs are inside their respective class within the file astromatic.py

In [ ]:
import os
import sys

import glob
import logging

from astropy.io import fits

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_tools
from pycavi import pycavi_astromatic_tools

In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/'+'../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
project_dir = config['Directories']['project_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
reduced_std_dir = config['Directories']['reduced_std_dir']
log_dir = config['Directories']['project_logs_dir']

# here we select all the folders with the _pre_wcs.fits mosaic 

#targets_to_scamp = glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.fits")
targets_to_scamp_sci = glob.glob(reduced_science_dir+"/**/*_pre/*_wcs.fits", recursive=True)
targets_to_scamp_std = glob.glob(reduced_std_dir+"/**/*_pre/*_wcs.fits", recursive=True)
targets_to_scamp = targets_to_scamp_sci + targets_to_scamp_std
targets_to_scamp.sort()

c = 0 
redo_scamp = False
for target in targets_to_scamp:
    c += 1

    # we check if the head file exists. It it does we skip
    #
    head_file = target[:-5]+'.head'
    cat_file = target[:-5]+'.cat'
    print("Scamp head file: {}".format(head_file))
    if os.path.isfile(head_file) and not redo_scamp:
        print("Scamp head file exists: {}. Skipping".format(head_file))
        continue
    elif os.path.isfile(head_file) and redo_scamp:
        os.remove(head_file)
        if os.path.isfile(cat_file):
            os.remove(cat_file)
        
    # creates the Scamp folder where the log files will be located
    scamp_dir_path = "/".join(target.split("/")[:-4])+'/'+config['Directories']['scamp_dir_name']
    print("scamp_dir_path: {}".format(scamp_dir_path))
    
    try:
        logging.info("Atempting to create the Scamp_dir folder")
        os.mkdir(scamp_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")

    log_file_scamp_sextractor = scamp_dir_path + "/pymeffpp_scamp_sextractor.log"
    
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                                  logging.FileHandler(log_file_scamp_sextractor),
                                  logging.StreamHandler()])
    logging.getLogger().setLevel(logging.INFO)

    logging.info("--------------------------------------------------------------------------------------")
    logging.info("Preparing to Sextract and Scamp fits image : {:d}/{:d}".format(c, len(targets_to_scamp)))
    logging.info("Fits image : {}".format(target.split("/")[-1]))
    
    wcs_image_path =  '/'.join(target.split("/")[:-1])

    # here we count backward as the project directory could be any
    target_name = target.split('/')[-5]
    filter_name = target.split('/')[-4]
    logging.info("             _target: {}".format(target))
    logging.info("               path: {}".format('reduced_science_dir + target_dir + /' + '/'.join(target.split("/")[9:-1])))
    logging.info("             target: {}".format(target_name))
    logging.info("             filter: {}".format(filter_name))
    
    # SOURCE-EXTRACTOR ON OUTPUT OF SOLVE-FIELD
    #
    # we create the folders needed by sextractor
    # now the folder are created, it is moment to run sextractor first
    # first move the folder where the data are. This changing of directories
    # is needed because sorce-extractor cannot handle very long paths in its
    # input file
    #
    #sextractor_dir_path = reduced_science_dir+target_name+'/'+config['Directories']['sex_dir_name']
    sextractor_dir_path = "/".join(target.split("/")[:-4])+'/'+config['Directories']['sex_dir_name']
    try:
        logging.info("Atempting to create the Sextractor_dir folder: {}".format(reduced_science_dir+target_name))
        os.mkdir(sextractor_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")
        
    # This flag can safely left to False unless you decide otherwise.
    # It can save some time.
    #
    redo_sextractor = True
    ready_to_sextract = pycavi_astromatic_tools.prepare_to_sextract(target, redo_sextractor)

    # we tune some parameters for sextractr
    special_pars = {}
    special_pars["DETECT_MINAREA"] = 5       # minimum number of pixels above threshold
    special_pars["DETECT_MAXAREA"] = 0       # maximun number of pixels above threshold
    special_pars["DETECT_THRESH"] = 3.0      # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
    special_pars["ANALYSIS_THRESH"] = 3.0    # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
    special_pars["SEEING_FWHM"] = 1.2        # stellar FWHM in arcsec (we give here typical not so good value)
    special_pars["PHOT_APERTURES"] = 5        # MAG_APER aperture diameter(s) in pixels
    special_pars["MAG_ZEROPOINT"] = 25.0     # magnitude zero-point
    special_pars["GAIN"] =  1.0              # detector gain in e-/ADU - if DN in e/s then 1.0
    special_pars["MEMORY_OBJSTACK"] = 6000   # number of objects in stack [3000]
    special_pars["MEMORY_PIXSTACK"] = 600000 # number of pixels in stack [300000]
    
    if ready_to_sextract:
        pycavi_astromatic_tools.run_sextractor_on_target(target[:-5]+".fits", sextractor_dir_path, special_pars)
    
    # sextractor is done
    logging.info("SExtractor finished on image {}".format(target.split("/")[-1]))

    # SCAMP ON OUTPUT OF SOLVE WITH THE SOURCE_EXTRACTOR CATALOGUE
    #
    # At this point Sextractor is finished and we can start to run Scamp
    # check if the Scamp .head file exists, same procedure as for Sextractor
    #scamp_dir_path = reduced_science_dir+target_name+'/Scamp_dir/'
    try:
        logging.info("Atempting to create the Scamp_dir folder: {}".format(scamp_dir_path))
        os.mkdir(scamp_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")
    scamp_head_file = target[:-5]+".head"
    
    # At this point Sextractor is finished and we can start to run Scamp
    # check if the Scamp .head file exists, same procedure as for Sextractor

    # This flag must be kept True because the script fails if it finds 
    # the file from a previous run.
    redo_scamp = True
    ready_to_scamp = pycavi_astromatic_tools.prepare_to_scamp(target, redo_scamp)

    # Parameters that might be useful to adjust. This set assumes that solvefield
    # worked correctly.
    special_pars = {}    
    special_pars["ASTREF_CATALOG"] = "GAIA-DR2"       # NONE,FILE,USNO-A2,USNO-B1,GSC-2.3,
                                                      # TYCHO-2,UCAC-4,URAT-1,NOMAD-1,PPMX,
                                                      # CMC-15,2MASS,DENIS-3,SDSS-R9,SDSS-R12,
                                                      # IGSL,GAIA-DR1,GAIA-DR2,GAIA-EDR3,
                                                      # PANSTARRS-1, or ALLWISE
    special_pars["ASTREF_BAND"] = "REDDEST"           # Photom. band for astr.ref.magnitudes
                                                      # or DEFAULT, BLUEST, or REDDEST
    special_pars["ASTREFMAG_LIMITS"] = [-99.0, 99.0]  # Select magnitude range in ASTREF_BAND
    special_pars["PIXSCALE_MAXERR"] = 1.1             # Max scale-factor uncertainty
    special_pars["POSITION_MAXERR"] = 0.05            # Max positional uncertainty (arcmin)
    special_pars["MOSAIC_TYPE"] = "SAME_CRVAL"        # UNCHANGED, SAME_CRVAL, SHARE_PROJAXIS,
                                                      # FIX_FOCALPLANE or LOOSE
    special_pars["CROSSID_RADIUS"] = 2.0              # Cross-id initial radius (arcsec)
    special_pars["SN_THRESHOLDS"] = [5.0, 40.0]       # S/N thresholds (in sigmas) for all and high S/N sample
    special_pars["FWHM_THRESHOLDS"] = [3.0, 10.0]     # FWHM thresholds (in pixels) for sources
    special_pars["POSANGLE_MAXERR"] = 2.5             # Max position-angle uncertainty (deg)

    if ready_to_scamp:
        pycavi_astromatic_tools.run_scamp_on_target(target, scamp_dir_path, special_pars)

    
logging.info('\n\n -------------- DONE ---------------\n\n')  

In [ ]:
# Old sextractor configuration, If needed to change, please go to the SExtractor.write_default_param function in
# mefpp_astromatic.py file, and replace the lines by these ones (line 121 within code).

lines = """NUMBER                   Running object number                                    
EXT_NUMBER               FITS extension number                                    
FLUX_ISO               #  Isophotal flux                                            [count]
FLUXERR_ISO            #  RMS error for isophotal flux                              [count]
MAG_ISO                #  Isophotal magnitude                                       [mag]
MAGERR_ISO             #  RMS error for isophotal magnitude                         [mag]
FLUX_APER              #  Flux vector within fixed circular aperture(s)             [count]
FLUXERR_APER           #  RMS error vector for aperture flux(es)                    [count]
MAG_APER               #  Fixed aperture magnitude vector                           [mag]
MAGERR_APER            #  RMS error vector for fixed aperture mag.                  [mag]
FLUX_AUTO              ##  Flux within a Kron-like elliptical aperture               [count]
FLUXERR_AUTO           #  RMS error for AUTO flux                                   [count]
MAG_AUTO               #  Kron-like elliptical aperture magnitude                   [mag]
MAGERR_AUTO            #  RMS error for AUTO magnitude                              [mag]
A_IMAGE                  Profile RMS along major axis                              [pixel]
B_IMAGE                  Profile RMS along minor axis                              [pixel]
THETA_IMAGE              Position angle (CCW/x)                                    [deg]
XWIN_IMAGE               Windowed position estimate along x                        [pixel]
YWIN_IMAGE               Windowed position estimate along y                        [pixel]
ERRX2WIN_IMAGE           Variance of windowed pos along x                          [pixel**2]
ERRY2WIN_IMAGE           Variance of windowed pos along y                          [pixel**2]
ERRXYWIN_IMAGE           Covariance of windowed pos between x and y                [pixel**2]
ERRX2WIN_WORLD           Variance of windowed pos along X-WORLD (alpha)            [deg**2]
ERRY2WIN_WORLD           Variance of windowed pos along Y-WORLD (delta)            [deg**2]
ERRXYWIN_WORLD           Covariance of windowed pos X-WORLD/Y-WORLD                [deg**2]
ERRAWIN_IMAGE            RMS windowed pos error along major axis                   [pixel]
ERRBWIN_IMAGE            RMS windowed pos error along minor axis                   [pixel]
ERRTHETAWIN_IMAGE        Windowed error ellipse pos angle (CCW/x)                  [deg]
FLAGS                    Extraction flags                                         
FLAGS_WEIGHT             Weighted extraction flags                                
FLAGS_WIN                Flags for WINdowed parameters                            
FWHM_IMAGE               FWHM assuming a gaussian core                             [pixel]
ELONGATION               A_IMAGE/B_IMAGE                                          
ELLIPTICITY              1 - B_IMAGE/A_IMAGE                                      
CLASS_STAR                S/G classifier output                                    
FLUX_RADIUS            #  Fraction-of-light radii                                   [pixel]"""